In [ ]:
"""
----
캐글 원본(220장) + AI Hub 경구약제 데이터를 병합해 dataset을 만든다.


파싱: annotation 34,786건 (실패 138건) -> 정제 후 34,737건, 이미지 8,989장
파싱: annotation 40,233건 (실패 149건) -> 정제 후 40,178건, 이미지 10,495장
겹침: 7,489장 / 팀원1만 있는 것: 3,006장 / 팀원2만 있는 것: 1,500장
    (1,500 + 7,489 + 3,006 = 11,995 = 합쳐진 최종 이미지 수)
    캐글 병합 후 중복 제거: 최종 train 11,995장 / val 44장
    MD5 중복 검증: 0건 / 이미지-라벨 불일치: 0건

"""

In [ ]:
import os
import json
import shutil
import hashlib

import numpy as np
import cv2
import pandas as pd
from PIL import Image

In [ ]:
CLASS_MAP_PATH = "./data/class_map_118.json"

SUNGYUN_BASE_PATH = "./data/경구약제조합_5000종_이미지"                        # 순균님 원본
HARU_ANNOTATIONS_PATH = "./data/aihub_data/aihub_data/aihub_annotations"      # 하루님 annotation
HARU_IMAGES_ROOT = "./data/aihub_data/aihub_data/aihub_images"                # 하루님 이미지 (공식 AI Hub 배포 구조)

KAGGLE_DATASET_PATH = "./data/dataset"
COMBINED_PATH_FINAL = "./data/dataset_combined_final"

IMG_WIDTH, IMG_HEIGHT = 976, 1280
PAD_SIZE = 1280
FINAL_SIZE = 640
PAD_LEFT = (PAD_SIZE - IMG_WIDTH) // 2

In [ ]:
def parse_sungyun():
    contents = os.listdir(SUNGYUN_BASE_PATH)
    img_folders = [f for f in contents if not f.endswith("_json")]
    json_folders = [f for f in contents if f.endswith("_json")]

    img_names = set(img_folders)
    json_names = set(f.replace("_json", "") for f in json_folders)
    usable_folders = img_names & json_names
    print(f"사용 가능한 조합 수: {len(usable_folders)}")

    all_annotations = []
    success_count, fail_count = 0, 0

    for folder_name in usable_folders:
        json_folder = os.path.join(SUNGYUN_BASE_PATH, f"{folder_name}_json")
        for sub_folder_name in os.listdir(json_folder):
            sub_folder_path = os.path.join(json_folder, sub_folder_name)
            for json_fname in os.listdir(sub_folder_path):
                json_path = os.path.join(sub_folder_path, json_fname)
                try:
                    with open(json_path, "r", encoding="utf-8") as f:
                        d = json.load(f)
                    img_info = d["images"][0]
                    file_name = img_info["file_name"]
                    dl_name = img_info["dl_name"]
                    dl_mapping_code = img_info["dl_mapping_code"]
                    for ann in d.get("annotations", []):
                        all_annotations.append({
                            "file_name": file_name,
                            "dl_name": dl_name,
                            "dl_mapping_code": dl_mapping_code,
                            "bbox_x": ann["bbox"][0],
                            "bbox_y": ann["bbox"][1],
                            "bbox_w": ann["bbox"][2],
                            "bbox_h": ann["bbox"][3],
                        })
                    success_count += 1
                except Exception:
                    fail_count += 1

    print(f"성공: {success_count}, 실패: {fail_count}")

    df_aihub_new = pd.DataFrame(all_annotations)

    with open(CLASS_MAP_PATH, "r", encoding="utf-8") as f:
        class_map = json.load(f)
    name_to_idx = class_map["name_to_idx"]
    df_aihub_new["category_id"] = df_aihub_new["dl_name"].map(name_to_idx)

    df_aihub_final = df_aihub_new[
        ["file_name", "dl_name", "category_id", "bbox_x", "bbox_y", "bbox_w", "bbox_h"]
    ].copy()
    df_aihub_final["category_id"] = df_aihub_final["category_id"].astype(int)

    print(f"최종 annotation 수: {len(df_aihub_final)}")
    df_aihub_final.to_csv("./data/aihub_sungkyun_merged.csv", index=False, encoding="utf-8-sig")
    return df_aihub_final, usable_folders

In [ ]:
def parse_haru():
    ann_contents = os.listdir(HARU_ANNOTATIONS_PATH)

    img_items = os.listdir(HARU_IMAGES_ROOT)
    ts_folder = os.path.join(HARU_IMAGES_ROOT, img_items[1])
    vl_path = os.path.join(ts_folder, os.listdir(ts_folder)[0])
    training_path = os.path.join(vl_path, "1.Training")
    source_path = os.path.join(training_path, os.listdir(training_path)[1])
    final_img_path = os.path.join(source_path, os.listdir(source_path)[0])

    img_folders_haru = os.listdir(final_img_path)
    img_folders_haru_clean = [f for f in img_folders_haru if not f.endswith(".cache")]

    ann_names_haru = set(f.replace("_json", "") for f in ann_contents if f.endswith("_json"))
    img_names_haru = set(img_folders_haru_clean)

    both_haru = img_names_haru & ann_names_haru
    print(f"둘 다 있는 조합: {len(both_haru)}")

    all_annotations_haru = []
    success_count, fail_count = 0, 0

    for folder_name in both_haru:
        json_folder = os.path.join(HARU_ANNOTATIONS_PATH, f"{folder_name}_json")
        for sub_folder_name in os.listdir(json_folder):
            sub_folder_path = os.path.join(json_folder, sub_folder_name)
            if not os.path.isdir(sub_folder_path) or sub_folder_name.startswith("."):
                continue
            for json_fname in os.listdir(sub_folder_path):
                if json_fname.startswith(".") or not json_fname.endswith(".json"):
                    continue
                json_path = os.path.join(sub_folder_path, json_fname)
                try:
                    with open(json_path, "r", encoding="utf-8") as f:
                        d = json.load(f)
                    img_info = d["images"][0]
                    file_name = img_info["file_name"]
                    dl_name = img_info["dl_name"]
                    dl_mapping_code = img_info["dl_mapping_code"]
                    for ann in d.get("annotations", []):
                        all_annotations_haru.append({
                            "file_name": file_name,
                            "dl_name": dl_name,
                            "dl_mapping_code": dl_mapping_code,
                            "bbox_x": ann["bbox"][0],
                            "bbox_y": ann["bbox"][1],
                            "bbox_w": ann["bbox"][2],
                            "bbox_h": ann["bbox"][3],
                        })
                    success_count += 1
                except Exception:
                    fail_count += 1

    print(f"성공: {success_count}, 실패: {fail_count}")

    df_haru = pd.DataFrame(all_annotations_haru)
    with open(CLASS_MAP_PATH, "r", encoding="utf-8") as f:
        class_map = json.load(f)
    name_to_idx = class_map["name_to_idx"]
    df_haru["category_id"] = df_haru["dl_name"].map(name_to_idx)

    print(f"고유 이미지 수: {df_haru['file_name'].nunique()}")
    df_haru.to_csv("./data/aihub_haru_merged.csv", index=False, encoding="utf-8-sig")
    return df_haru, final_img_path, both_haru

In [ ]:
def clean_bbox_range(df_aihub_final, df_haru):
    df_aihub_final_clean = df_aihub_final[
        (df_aihub_final["bbox_x"] >= 0)
        & (df_aihub_final["bbox_y"] >= 0)
        & (df_aihub_final["bbox_x"] + df_aihub_final["bbox_w"] <= IMG_WIDTH)
        & (df_aihub_final["bbox_y"] + df_aihub_final["bbox_h"] <= IMG_HEIGHT)
    ]
    df_haru_clean = df_haru[
        (df_haru["bbox_x"] >= 0)
        & (df_haru["bbox_y"] >= 0)
        & (df_haru["bbox_x"] + df_haru["bbox_w"] <= IMG_WIDTH)
        & (df_haru["bbox_y"] + df_haru["bbox_h"] <= IMG_HEIGHT)
    ]
    print(f"{len(df_aihub_final)} -> {len(df_aihub_final_clean)} (제거 {len(df_aihub_final) - len(df_aihub_final_clean)}건)")
    print(f"{len(df_haru)} -> {len(df_haru_clean)} (제거 {len(df_haru) - len(df_haru_clean)}건)")

    df_aihub_final_clean.to_csv("./data/aihub_sungkyun_clean.csv", index=False, encoding="utf-8-sig")
    df_haru_clean.to_csv("./data/aihub_haru_clean.csv", index=False, encoding="utf-8-sig")
    return df_aihub_final_clean, df_haru_clean

In [ ]:
def match_images(df_aihub_final_clean, usable_folders, df_haru_clean, final_img_path, both_haru):
    image_path_map = {}
    for folder_name in usable_folders:
        img_folder = os.path.join(SUNGYUN_BASE_PATH, folder_name)
        for fname in os.listdir(img_folder):
            if fname.endswith(".png") and "_index" not in fname:
                image_path_map[fname] = os.path.join(img_folder, fname)

    matched_sk_clean = set(df_aihub_final_clean["file_name"].unique()) & set(image_path_map.keys())
    print(f"찾은 이미지: {len(image_path_map)}")
    print(f"정제된 annotation의 고유 이미지: {df_aihub_final_clean['file_name'].nunique()}")
    print(f"이미지+정제annotation 매칭: {len(matched_sk_clean)}")

    image_path_map_haru = {}
    for folder_name in both_haru:
        img_folder = os.path.join(final_img_path, folder_name)
        if not os.path.isdir(img_folder):
            continue
        for fname in os.listdir(img_folder):
            if fname.endswith(".png") and "_index" not in fname:
                image_path_map_haru[fname] = os.path.join(img_folder, fname)

    matched_haru_clean = set(df_haru_clean["file_name"].unique()) & set(image_path_map_haru.keys())
    print(f"\n찾은 이미지: {len(image_path_map_haru)}")
    print(f"정제된 annotation의 고유 이미지: {df_haru_clean['file_name'].nunique()}")
    print(f"이미지+정제annotation 매칭: {len(matched_haru_clean)}")

    overlap_clean = matched_sk_clean & matched_haru_clean
    only_haru_clean = matched_haru_clean - matched_sk_clean
    only_sk_clean = matched_sk_clean - matched_haru_clean

    print(f"\n겹치는 것: {len(overlap_clean)}")
    print(f"추가할 것: {len(only_haru_clean)}")
    print(f"{len(only_sk_clean)}")
    print(f"검증: {len(only_sk_clean)} + {len(overlap_clean)} + {len(only_haru_clean)} = "
          f"{len(only_sk_clean) + len(overlap_clean) + len(only_haru_clean)}")

    return image_path_map, matched_sk_clean, image_path_map_haru, only_haru_clean

In [ ]:
def add_padding_and_resize(src_path, dst_path):
    stream = np.fromfile(src_path, dtype=np.uint8)
    img_cv = cv2.imdecode(stream, cv2.IMREAD_COLOR)
    img_cv = cv2.cvtColor(img_cv, cv2.COLOR_BGR2RGB)
    img = Image.fromarray(img_cv)

    padded = Image.new("RGB", (PAD_SIZE, PAD_SIZE), (114, 114, 114))
    padded.paste(img, (PAD_LEFT, 0))
    resized = padded.resize((FINAL_SIZE, FINAL_SIZE))
    resized.save(dst_path)


def convert_to_yolo(bbox_x, bbox_y, bbox_w, bbox_h, class_idx):
    x_center = (bbox_x + PAD_LEFT + bbox_w / 2) / PAD_SIZE
    y_center = (bbox_y + bbox_h / 2) / PAD_SIZE
    w = bbox_w / PAD_SIZE
    h = bbox_h / PAD_SIZE
    return f"{class_idx} {x_center:.6f} {y_center:.6f} {w:.6f} {h:.6f}"

In [ ]:
def convert_and_save_sungyun(df_aihub_final_clean, matched_sk_clean, image_path_map):
    success_sk, fail_sk = 0, 0
    for file_name in matched_sk_clean:
        src = image_path_map[file_name]
        dst = f"{COMBINED_PATH_FINAL}/images/train/aihub2_{file_name}"
        try:
            add_padding_and_resize(src, dst)
        except Exception:
            fail_sk += 1
            continue
        rows = df_aihub_final_clean[df_aihub_final_clean["file_name"] == file_name]
        label_lines = [
            convert_to_yolo(row["bbox_x"], row["bbox_y"], row["bbox_w"], row["bbox_h"], row["category_id"])
            for _, row in rows.iterrows()
        ]
        label_path = f"{COMBINED_PATH_FINAL}/labels/train/aihub2_{file_name.replace('.png', '.txt')}"
        with open(label_path, "w") as f:
            f.write("\n".join(label_lines))
        success_sk += 1
    print(f"변환 완료: {success_sk}장 성공, {fail_sk}장 실패")


def convert_and_save_haru(df_haru_clean, only_haru_clean, image_path_map_haru):
    success_haru_final, fail_haru_final = 0, 0
    for file_name in only_haru_clean:
        src = image_path_map_haru[file_name]
        dst = f"{COMBINED_PATH_FINAL}/images/train/haru_{file_name}"
        try:
            add_padding_and_resize(src, dst)
        except Exception:
            fail_haru_final += 1
            continue
        rows = df_haru_clean[df_haru_clean["file_name"] == file_name]
        label_lines = [
            convert_to_yolo(row["bbox_x"], row["bbox_y"], row["bbox_w"], row["bbox_h"], int(row["category_id"]))
            for _, row in rows.iterrows()
        ]
        label_path = f"{COMBINED_PATH_FINAL}/labels/train/haru_{file_name.replace('.png', '.txt')}"
        with open(label_path, "w") as f:
            f.write("\n".join(label_lines))
        success_haru_final += 1
    print(f"변환 완료: {success_haru_final}장 성공, {fail_haru_final}장 실패")

In [ ]:
def merge_kaggle():
    for fname in os.listdir(f"{KAGGLE_DATASET_PATH}/images/train"):
        shutil.copy(f"{KAGGLE_DATASET_PATH}/images/train/{fname}",
                    f"{COMBINED_PATH_FINAL}/images/train/kaggle_{fname}")
    for fname in os.listdir(f"{KAGGLE_DATASET_PATH}/labels/train"):
        shutil.copy(f"{KAGGLE_DATASET_PATH}/labels/train/{fname}",
                    f"{COMBINED_PATH_FINAL}/labels/train/kaggle_{fname}")

    for fname in os.listdir(f"{KAGGLE_DATASET_PATH}/images/val"):
        shutil.copy(f"{KAGGLE_DATASET_PATH}/images/val/{fname}",
                    f"{COMBINED_PATH_FINAL}/images/val/{fname}")
    for fname in os.listdir(f"{KAGGLE_DATASET_PATH}/labels/val"):
        shutil.copy(f"{KAGGLE_DATASET_PATH}/labels/val/{fname}",
                    f"{COMBINED_PATH_FINAL}/labels/val/{fname}")

    print("캐글 데이터 복사 완료")
    print(f"최종 train 이미지 수: {len(os.listdir(f'{COMBINED_PATH_FINAL}/images/train'))}")
    print(f"최종 val 이미지 수: {len(os.listdir(f'{COMBINED_PATH_FINAL}/images/val'))}")

In [ ]:
def dedup_kaggle_vs_aihub():
    train_dir = f"{COMBINED_PATH_FINAL}/images/train"
    label_dir = f"{COMBINED_PATH_FINAL}/labels/train"

    kaggle_files_orig = set()
    for f in os.listdir(train_dir):
        if f.startswith("kaggle_"):
            kaggle_files_orig.add(f.split("_", 1)[1])
    print(f"캐글 이미지 수: {len(kaggle_files_orig)}")

    removed_aihub_count = 0
    for f in list(os.listdir(train_dir)):
        if f.startswith("aihub2_") or f.startswith("haru_"):
            original = f.split("_", 1)[1]
            if original in kaggle_files_orig:
                os.remove(f"{train_dir}/{f}")
                label_f = f.replace(".png", ".txt")
                label_path = f"{label_dir}/{label_f}"
                if os.path.exists(label_path):
                    os.remove(label_path)
                removed_aihub_count += 1

    print(f"AI Hub 쪽 중복 파일 제거: {removed_aihub_count}개")
    print(f"최종 train 이미지 수: {len(os.listdir(train_dir))}")

In [ ]:
def get_hash(filepath):
    with open(filepath, "rb") as f:
        return hashlib.md5(f.read()).hexdigest()


def final_verification():
    train_dir_verify = f"{COMBINED_PATH_FINAL}/images/train"
    all_files_verify = os.listdir(train_dir_verify)
    hash_to_files_verify = {}

    for i, fname in enumerate(all_files_verify):
        h = get_hash(f"{train_dir_verify}/{fname}")
        hash_to_files_verify.setdefault(h, []).append(fname)
        if (i + 1) % 3000 == 0:
            print(f"{i + 1}/{len(all_files_verify)} 확인 중...")

    duplicates_verify = {h: files for h, files in hash_to_files_verify.items() if len(files) > 1}
    print(f"\n전체 이미지: {len(all_files_verify)}장")
    print(f"고유 이미지: {len(hash_to_files_verify)}장")
    print(f"중복 그룹: {len(duplicates_verify)}개")

    train_imgs_verify = set(f.replace(".png", "") for f in os.listdir(f"{COMBINED_PATH_FINAL}/images/train"))
    train_labels_verify = set(f.replace(".txt", "") for f in os.listdir(f"{COMBINED_PATH_FINAL}/labels/train"))
    print(f"\n이미지: {len(train_imgs_verify)}, 라벨: {len(train_labels_verify)}")
    print(f"불일치: {len(train_imgs_verify.symmetric_difference(train_labels_verify))}건")


In [ ]:
def write_data_yaml():
    import yaml
    with open(CLASS_MAP_PATH, "r", encoding="utf-8") as f:
        class_map = json.load(f)

    data_yaml = {
        "path": os.path.abspath(COMBINED_PATH_FINAL),
        "train": "images/train",
        "val": "images/val",
        "nc": 118,
        "names": [class_map["idx_to_name"][str(i)] for i in range(118)],
    }
    yaml_path_final = f"{COMBINED_PATH_FINAL}/data.yaml"
    with open(yaml_path_final, "w", encoding="utf-8") as f:
        yaml.dump(data_yaml, f, allow_unicode=True, sort_keys=False)
    print("data.yaml 생성 완료")
    print(f"저장 경로: {yaml_path_final}")

In [ ]:
os.makedirs(f"{COMBINED_PATH_FINAL}/images/train", exist_ok=True)
os.makedirs(f"{COMBINED_PATH_FINAL}/labels/train", exist_ok=True)
os.makedirs(f"{COMBINED_PATH_FINAL}/images/val", exist_ok=True)
os.makedirs(f"{COMBINED_PATH_FINAL}/labels/val", exist_ok=True)

df_aihub_final, usable_folders = parse_sungyun()
df_haru, final_img_path, both_haru = parse_haru()

df_aihub_final_clean, df_haru_clean = clean_bbox_range(df_aihub_final, df_haru)

image_path_map, matched_sk_clean, image_path_map_haru, only_haru_clean = match_images(
    df_aihub_final_clean, usable_folders, df_haru_clean, final_img_path, both_haru
)

convert_and_save_sungyun(df_aihub_final_clean, matched_sk_clean, image_path_map)
convert_and_save_haru(df_haru_clean, only_haru_clean, image_path_map_haru)

merge_kaggle()
dedup_kaggle_vs_aihub()
final_verification()
write_data_yaml()